# 核函数：把非线性问题搬到高维空间

在机器学习中，核函数用于衡量样本之间的相似性。它的核心价值在于：不显式构造高维特征，而是通过核技巧直接在原空间完成高维内积计算。

---

## 1. 核函数的定义

### 数学形式
如果存在一个映射 $\phi(x)$，使得：
$$K(x, z) = \phi(x)^T \phi(z)$$
那么 $K(x, z)$ 就称为核函数。

### 直观理解
核函数的作用是：
- 输入空间中的两个样本，先不显式映射到高维；
- 直接计算它们在高维特征空间中的内积；
- 从而让线性算法间接处理非线性问题。

### 生动比喻
核函数像是**地图上的导航软件**。你不需要自己背一张完整的高维地图，软件已经帮你算好了路线和距离。你关心的是结果，而不是中间的全部坐标展开。

### 常见核函数
- 线性核：$K(x, z) = x^T z$
- 多项式核：$K(x, z) = (x^T z + c)^d$
- 高斯 RBF 核：$K(x, z) = \exp(-\gamma \|x-z\|^2)$

## 2. 手写实现：核函数计算

### 小数据实例
设两个二维样本：
$$x = [1, 2], \quad z = [3, 4]$$
先计算它们的普通内积：
$$x^T z = 1 \times 3 + 2 \times 4 = 11$$

如果采用二阶多项式核，并取 $c=1$：
$$K(x, z) = (x^T z + 1)^2 = (11 + 1)^2 = 144$$

如果采用高斯 RBF 核，并取 $\gamma = 0.5$：
$$\|x-z\|^2 = (1-3)^2 + (2-4)^2 = 8$$
$$K(x, z) = \exp(-0.5 \times 8) = e^{-4}$$

这个结果说明：
- 多项式核会把样本之间的交互项增强；
- RBF 核会随着距离变大而快速衰减。

In [1]:
import numpy as np

def linear_kernel_manual(x, z):
    """手写线性核：x^T z"""
    return np.dot(x, z)

def polynomial_kernel_manual(x, z, degree=2, coef0=1.0):
    """手写多项式核：(x^T z + coef0)^degree"""
    return (np.dot(x, z) + coef0) ** degree

def rbf_kernel_manual(x, z, gamma=0.5):
    """手写高斯 RBF 核：exp(-gamma * ||x-z||^2)"""
    diff = x - z
    return np.exp(-gamma * np.dot(diff, diff))

x = np.array([1.0, 2.0])
z = np.array([3.0, 4.0])

print("线性核:", linear_kernel_manual(x, z))
print("二阶多项式核:", polynomial_kernel_manual(x, z, degree=2, coef0=1.0))
print("RBF 核:", rbf_kernel_manual(x, z, gamma=0.5))

线性核: 11.0
二阶多项式核: 144.0
RBF 核: 0.01831563888873418


## 3. 手写核矩阵

### 定义
给定样本集 $X = \{x_1, x_2, ..., x_n\}$，核矩阵定义为：
$$K_{ij} = K(x_i, x_j)$$
它是一个对称矩阵，描述了样本两两之间的相似性。

### 小数据实例
设有三个二维样本：
$$X = \begin{bmatrix} 1 & 2 \\ 2 & 1 \\ 3 & 4 \end{bmatrix}$$
我们用二阶多项式核构造核矩阵。

In [2]:
def manual_kernel_matrix(data, kernel_func, **kwargs):
    """手写核矩阵"""
    data = np.asarray(data, dtype=float)
    n_samples = data.shape[0]
    kernel_matrix = np.zeros((n_samples, n_samples), dtype=float)

    for i in range(n_samples):
        for j in range(n_samples):
            kernel_matrix[i, j] = kernel_func(data[i], data[j], **kwargs)

    return kernel_matrix


X_small = np.array([
    [1.0, 2.0],
    [2.0, 1.0],
    [3.0, 4.0],
])

K_poly = manual_kernel_matrix(X_small, polynomial_kernel_manual, degree=2, coef0=1.0)
print("小数据样本:\n", X_small)
print("二阶多项式核矩阵:\n", K_poly)

小数据样本:
 [[1. 2.]
 [2. 1.]
 [3. 4.]]
二阶多项式核矩阵:
 [[ 36.  25. 144.]
 [ 25.  36. 121.]
 [144. 121. 676.]]


## 4. 库函数调用：Sklearn 核函数对比

Scikit-learn 提供了现成的核函数工具，可以直接计算线性核、多项式核和 RBF 核。

In [3]:
from sklearn.metrics.pairwise import linear_kernel, polynomial_kernel, rbf_kernel

K_linear_sklearn = linear_kernel(X_small)
K_poly_sklearn = polynomial_kernel(X_small, degree=2, coef0=1.0)
K_rbf_sklearn = rbf_kernel(X_small, gamma=0.5)

print("Sklearn 线性核矩阵:\n", K_linear_sklearn)
print("Sklearn 二阶多项式核矩阵:\n", K_poly_sklearn)
print("Sklearn RBF 核矩阵:\n", K_rbf_sklearn)
print("手写与 Sklearn 的多项式核是否一致:", np.allclose(K_poly, K_poly_sklearn))

Sklearn 线性核矩阵:
 [[ 5.  4. 11.]
 [ 4.  5. 10.]
 [11. 10. 25.]]
Sklearn 二阶多项式核矩阵:
 [[ 12.25   9.    42.25]
 [  9.    12.25  36.  ]
 [ 42.25  36.   182.25]]
Sklearn RBF 核矩阵:
 [[1.         0.36787944 0.01831564]
 [0.36787944 1.         0.00673795]
 [0.01831564 0.00673795 1.        ]]
手写与 Sklearn 的多项式核是否一致: False


## 5. 核技巧与 SVM 的关系

支持向量机在处理非线性分类问题时，常常通过核函数把样本映射到高维空间，再在高维空间中寻找最大间隔超平面。

### 关键点
- 不是显式构造高维特征；
- 只计算样本之间的核矩阵；
- 训练复杂度和表达能力之间需要平衡。

### 过渡理解
如果线性可分问题可以直接用线性模型解决，那么核函数更适合处理“边界弯曲”的非线性数据。

### 练习与思考
1. **计算题**：给定 $x=[2, 1]$ 和 $z=[1, 3]$，分别计算线性核和二阶多项式核。
2. **理解题**：为什么核函数可以避免显式计算高维特征？
3. **应用题**：RBF 核中的 $\gamma$ 变大或变小，会如何影响模型的决策边界？